# EDA Forense Jurídica — Processos e Eventos

Este notebook conduz uma EDA forense sobre duas bases:

1. **Base de processos**: uma linha por processo/protesto, com dados estruturados, partes, assunto, valores, andamento, resultado, sinais e predições externas.
2. **Base de eventos**: várias linhas por processo, com a sequência de eventos/movimentos processuais.

O objetivo é validar a qualidade das bases, entender o grão dos dados, investigar nulos/cardinalidade/leakage, categorizar eventos processuais e gerar uma primeira ABT exploratória com uma linha por processo.

> Pré-requisito: o arquivo `eda_forense_juridico.py` deve estar na raiz do projeto.

## 0. Estrutura esperada do projeto

```text
jurimetria-mvp/
│
├── eda_forense_juridico.py
├── notebooks/
│   └── 01_eda_forense_juridico.ipynb
│
├── data/
│   ├── raw/
│   │   ├── processos.parquet
│   │   └── eventos.parquet
│   │
│   └── processed/
│
└── outputs/
    └── reports/
```

Se seus arquivos estiverem em CSV, altere os caminhos na seção de configuração.

## 1. Imports básicos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 2. Importar funções da EDA

In [ ]:
import sys
from pathlib import Path

# Se este notebook estiver dentro da pasta notebooks/, a raiz do projeto é '..'
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from eda_forense_juridico import (
    read_data,
    basic_overview,
    key_audit,
    process_event_relationship_audit,
    null_report,
    classify_null_severity,
    cardinality_report,
    dominant_value_report,
    numeric_summary,
    date_report,
    classify_column_by_name,
    identify_external_prediction_cols,
    identify_deprecated_cols,
    identify_link_cols,
    identify_long_text_cols,
    identify_result_or_post_decision_cols,
    recommend_column_usage,
    normalize_text,
    categorize_event_type,
    sample_event_types_by_category,
    build_basic_event_features,
    build_first_last_event_features,
    build_event_category_features,
    build_initial_target,
    target_rate_by_category,
    analyze_event_flags_vs_target,
)

print("Funções importadas com sucesso.")

### Diagnóstico caso o import falhe

In [ ]:
import os

print("Diretório atual:", os.getcwd())
print("Arquivos na raiz configurada:")
print(list(PROJECT_ROOT.glob("*"))[:20])

## 3. Configurações do notebook

In [ ]:
PROCESS_ID_COL = "numero_processo"

EVENT_NUM_COL = "numero"
EVENT_TYPE_COL = "tipo"
EVENT_TEXT_COL = "texto"
EVENT_DATE_COL = "data"

TARGET_COL = "perda_banco"

BASE_DIR = Path("..")  # ajuste para Path(".") se o notebook estiver na raiz do projeto
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Arquivos em Parquet
PROCESSOS_FILE = RAW_DIR / "processos.parquet"
EVENTOS_FILE = RAW_DIR / "eventos.parquet"

# Se estiver usando CSV, comente as linhas acima e descomente:
# PROCESSOS_FILE = RAW_DIR / "processos.csv"
# EVENTOS_FILE = RAW_DIR / "eventos.csv"

print("PROCESSOS_FILE:", PROCESSOS_FILE)
print("EVENTOS_FILE:", EVENTOS_FILE)

## 4. Carregar base de processos

In [ ]:
df_processos = read_data(PROCESSOS_FILE)

print(df_processos.shape)
df_processos.head()

In [ ]:
df_processos.columns.tolist()

In [ ]:
print("A coluna de chave existe?", PROCESS_ID_COL in df_processos.columns)

## 5. Carregar base de eventos

In [ ]:
event_cols = [
    PROCESS_ID_COL,
    EVENT_NUM_COL,
    EVENT_TYPE_COL,
    EVENT_TEXT_COL,
    EVENT_DATE_COL,
]

df_eventos = read_data(EVENTOS_FILE, columns=event_cols)

print(df_eventos.shape)
df_eventos.head()

In [ ]:
# Para CSV:
# sample_eventos = pd.read_csv(EVENTOS_FILE, nrows=5)
# sample_eventos.columns.tolist()

# Para Parquet:
# import pyarrow.parquet as pq
# parquet_file = pq.ParquetFile(EVENTOS_FILE)
# parquet_file.schema.names

## 6. Visão geral das bases

In [ ]:
overview_processos = basic_overview(df_processos, "processos")
overview_eventos = basic_overview(df_eventos, "eventos")

overview_processos

In [ ]:
overview_eventos

In [ ]:
overview_processos.to_csv(REPORTS_DIR / "overview_processos.csv", index=False)
overview_eventos.to_csv(REPORTS_DIR / "overview_eventos.csv", index=False)

## 7. Auditar chave da base de processos

In [ ]:
duplicados_processos = key_audit(
    df_processos,
    PROCESS_ID_COL,
    "processos"
)

duplicados_processos.head(20)

In [ ]:
duplicados_processos.to_csv(REPORTS_DIR / "duplicados_chave_processos.csv", index=False)

In [ ]:
if not duplicados_processos.empty:
    ids_duplicados = duplicados_processos[PROCESS_ID_COL].head(5).tolist()
    display(df_processos[df_processos[PROCESS_ID_COL].isin(ids_duplicados)].sort_values(PROCESS_ID_COL))
else:
    print("Não há chaves duplicadas na base de processos.")

## 8. Eventos por processo

In [ ]:
eventos_por_processo = (
    df_eventos
    .groupby(PROCESS_ID_COL)
    .size()
    .reset_index(name="qtd_eventos")
    .sort_values("qtd_eventos", ascending=False)
)

eventos_por_processo.head(20)

In [ ]:
eventos_por_processo["qtd_eventos"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
)

In [ ]:
eventos_por_processo.to_csv(REPORTS_DIR / "eventos_por_processo.csv", index=False)

## 9. Relação processos × eventos

In [ ]:
rel_audit, processos_sem_eventos, eventos_sem_processo = process_event_relationship_audit(
    df_processos,
    df_eventos,
    PROCESS_ID_COL
)

rel_audit

In [ ]:
rel_audit.to_csv(REPORTS_DIR / "auditoria_processos_eventos.csv", index=False)

pd.DataFrame({PROCESS_ID_COL: list(processos_sem_eventos)}).to_csv(
    REPORTS_DIR / "processos_sem_eventos.csv",
    index=False
)

pd.DataFrame({PROCESS_ID_COL: list(eventos_sem_processo)}).to_csv(
    REPORTS_DIR / "eventos_sem_processo.csv",
    index=False
)

## 10. Relatório de nulos dos processos

In [ ]:
null_processos = null_report(df_processos)
null_processos["faixa_nulos"] = null_processos["perc_nulos"].apply(classify_null_severity)

null_processos.head(50)

In [ ]:
null_processos.to_csv(REPORTS_DIR / "null_report_processos.csv", index=False)
null_processos["faixa_nulos"].value_counts()

## 11. Relatório de nulos dos eventos

In [ ]:
null_eventos = null_report(df_eventos)
null_eventos["faixa_nulos"] = null_eventos["perc_nulos"].apply(classify_null_severity)

null_eventos

In [ ]:
null_eventos.to_csv(REPORTS_DIR / "null_report_eventos.csv", index=False)

## 12. Cardinalidade da base de processos

In [ ]:
card_processos = cardinality_report(df_processos)

card_processos.head(50)

In [ ]:
card_processos.to_csv(REPORTS_DIR / "cardinality_report_processos.csv", index=False)

## 13. Cardinalidade dos eventos

In [ ]:
card_eventos = cardinality_report(df_eventos)

card_eventos

In [ ]:
card_eventos.to_csv(REPORTS_DIR / "cardinality_report_eventos.csv", index=False)

## 14. Valor dominante nos processos

In [ ]:
dominant_processos = dominant_value_report(df_processos)

dominant_processos.head(50)

In [ ]:
dominant_processos.to_csv(REPORTS_DIR / "dominant_report_processos.csv", index=False)

## 15. Resumo numérico dos processos

In [ ]:
numeric_processos = numeric_summary(df_processos)

numeric_processos.head(100)

In [ ]:
numeric_processos.to_csv(REPORTS_DIR / "numeric_summary_processos.csv", index=False)

## 16. Relatório de datas

In [ ]:
date_processos = date_report(df_processos)

date_processos.head(100)

In [ ]:
date_processos.to_csv(REPORTS_DIR / "date_report_processos.csv", index=False)

In [ ]:
date_eventos = date_report(df_eventos)

date_eventos

In [ ]:
date_eventos.to_csv(REPORTS_DIR / "date_report_eventos.csv", index=False)

## 17. Classificar colunas de processos por grupo jurídico

In [ ]:
column_groups = pd.DataFrame({
    "coluna": df_processos.columns
})

column_groups["grupo_inferido"] = column_groups["coluna"].apply(classify_column_by_name)

column_groups["grupo_inferido"].value_counts()

In [ ]:
column_groups.sort_values("grupo_inferido").head(100)

In [ ]:
column_groups.to_csv(REPORTS_DIR / "processos_column_groups.csv", index=False)

## 18. Identificar possíveis leakage e descartes

In [ ]:
external_pred_cols = identify_external_prediction_cols(df_processos.columns)
deprecated_cols = identify_deprecated_cols(df_processos.columns)
link_cols = identify_link_cols(df_processos.columns)
long_text_cols = identify_long_text_cols(df_processos.columns)
post_decision_cols = identify_result_or_post_decision_cols(df_processos.columns)

print("Predições externas:", len(external_pred_cols))
print("Deprecated:", len(deprecated_cols))
print("Links:", len(link_cols))
print("Textos longos:", len(long_text_cols))
print("Resultado/pós-decisão:", len(post_decision_cols))

In [ ]:
external_pred_cols[:50]

In [ ]:
post_decision_cols[:100]

In [ ]:
leakage_reference = pd.DataFrame({
    "coluna": (
        external_pred_cols
        + deprecated_cols
        + link_cols
        + long_text_cols
        + post_decision_cols
    )
}).drop_duplicates()

leakage_reference.to_csv(
    REPORTS_DIR / "colunas_candidatas_leakage_ou_remocao.csv",
    index=False
)

leakage_reference.head(100)

## 19. Perfil completo das colunas

In [ ]:
processos_profile = (
    null_processos
    .merge(
        card_processos[["coluna", "qtd_distintos", "perc_distintos"]],
        on="coluna",
        how="left"
    )
    .merge(
        dominant_processos[["coluna", "valor_mais_frequente", "perc_valor_mais_frequente"]],
        on="coluna",
        how="left"
    )
    .merge(
        column_groups,
        on="coluna",
        how="left"
    )
)

processos_profile["recomendacao_uso"] = processos_profile.apply(
    lambda row: recommend_column_usage(
        row,
        external_pred_cols=external_pred_cols,
        deprecated_cols=deprecated_cols,
        link_cols=link_cols,
        long_text_cols=long_text_cols,
        post_decision_cols=post_decision_cols,
    ),
    axis=1
)

processos_profile = processos_profile.sort_values(
    ["grupo_inferido", "perc_nulos"],
    ascending=[True, False]
)

processos_profile.head(100)

In [ ]:
processos_profile.to_csv(
    REPORTS_DIR / "processos_profile_com_recomendacao.csv",
    index=False
)

In [ ]:
processos_profile.query("recomendacao_uso == 'candidata_feature'").head(100)

In [ ]:
processos_profile.query("recomendacao_uso == 'usar_como_target_ou_remover_por_leakage'").head(100)

## 20. Tratar eventos: tipo, data e categoria

In [ ]:
df_eventos[EVENT_DATE_COL] = pd.to_datetime(df_eventos[EVENT_DATE_COL], errors="coerce")

df_eventos["tipo_clean"] = df_eventos[EVENT_TYPE_COL].map(normalize_text)

df_eventos["tipo_categoria"] = df_eventos["tipo_clean"].map(categorize_event_type)

df_eventos.head()

In [ ]:
df_eventos[[EVENT_TYPE_COL, "tipo_clean", "tipo_categoria"]].head(50)

In [ ]:
df_eventos.to_parquet(
    PROCESSED_DIR / "eventos_tratados_eda.parquet",
    index=False
)

## 21. Frequência dos tipos de evento

In [ ]:
event_type_freq = (
    df_eventos["tipo_clean"]
    .value_counts(dropna=False)
    .reset_index()
)

event_type_freq.columns = ["tipo_clean", "qtd_eventos"]
event_type_freq["perc_eventos"] = event_type_freq["qtd_eventos"] / len(df_eventos)
event_type_freq["perc_acumulado"] = event_type_freq["perc_eventos"].cumsum()

event_type_freq.head(100)

In [ ]:
event_type_freq.to_csv(
    REPORTS_DIR / "event_type_frequency_report.csv",
    index=False
)

## 22. Frequência das categorias de evento

In [ ]:
event_category_freq = (
    df_eventos["tipo_categoria"]
    .value_counts(dropna=False)
    .reset_index()
)

event_category_freq.columns = ["tipo_categoria", "qtd_eventos"]
event_category_freq["perc_eventos"] = event_category_freq["qtd_eventos"] / len(df_eventos)
event_category_freq["perc_acumulado"] = event_category_freq["perc_eventos"].cumsum()

event_category_freq

In [ ]:
event_category_freq.to_csv(
    REPORTS_DIR / "event_category_frequency_report.csv",
    index=False
)

## 23. Exemplos por categoria

In [ ]:
event_category_samples = sample_event_types_by_category(
    df_eventos,
    category_col="tipo_categoria",
    type_col="tipo_clean",
    n=20
)

event_category_samples.head(200)

In [ ]:
event_category_samples.to_csv(
    REPORTS_DIR / "event_category_samples.csv",
    index=False
)

## 24. Auditoria temporal dos eventos

In [ ]:
event_date_report = pd.DataFrame({
    "min_data_evento": [df_eventos[EVENT_DATE_COL].min()],
    "max_data_evento": [df_eventos[EVENT_DATE_COL].max()],
    "qtd_eventos": [len(df_eventos)],
    "qtd_eventos_sem_data": [df_eventos[EVENT_DATE_COL].isna().sum()],
    "perc_eventos_sem_data": [df_eventos[EVENT_DATE_COL].isna().mean()],
})

event_date_report

In [ ]:
event_date_report.to_csv(
    REPORTS_DIR / "event_date_report.csv",
    index=False
)

In [ ]:
df_eventos["ano_mes_evento"] = df_eventos[EVENT_DATE_COL].dt.to_period("M").astype(str)

events_by_month = (
    df_eventos
    .groupby("ano_mes_evento")
    .size()
    .reset_index(name="qtd_eventos")
    .sort_values("ano_mes_evento")
)

events_by_month.tail(30)

In [ ]:
events_by_month.to_csv(
    REPORTS_DIR / "events_by_month.csv",
    index=False
)

## 25. Criar features básicas de eventos

In [ ]:
df_eventos = df_eventos.sort_values(
    [PROCESS_ID_COL, EVENT_DATE_COL, EVENT_NUM_COL]
)

features_eventos_basicas = build_basic_event_features(df_eventos)

features_eventos_basicas.head()

In [ ]:
features_eventos_basicas.describe()

In [ ]:
features_eventos_basicas.to_parquet(
    PROCESSED_DIR / "features_eventos_basicas_eda.parquet",
    index=False
)

## 26. Primeiro, último e penúltimo evento

In [ ]:
features_primeiro_ultimo = build_first_last_event_features(df_eventos)

features_primeiro_ultimo.head()

In [ ]:
features_primeiro_ultimo.to_parquet(
    PROCESSED_DIR / "features_primeiro_ultimo_evento_eda.parquet",
    index=False
)

## 27. Contagem e flags por categoria de evento

In [ ]:
features_eventos_categoria = build_event_category_features(df_eventos)

features_eventos_categoria.head()

In [ ]:
features_eventos_categoria.to_parquet(
    PROCESSED_DIR / "features_eventos_categoria_eda.parquet",
    index=False
)

In [ ]:
features_eventos_categoria.columns.tolist()

## 28. Juntar features de eventos

In [ ]:
features_eventos = (
    features_eventos_basicas
    .merge(features_primeiro_ultimo, on=PROCESS_ID_COL, how="left")
    .merge(features_eventos_categoria, on=PROCESS_ID_COL, how="left")
)

print(features_eventos.shape)
features_eventos.head()

In [ ]:
features_eventos[PROCESS_ID_COL].nunique(), len(features_eventos)

In [ ]:
assert features_eventos[PROCESS_ID_COL].nunique() == len(features_eventos), "Features de eventos duplicaram processos!"

features_eventos.to_parquet(
    PROCESSED_DIR / "features_eventos_eda_inicial.parquet",
    index=False
)

## 29. Criar ABT exploratória

In [ ]:
df_abt_eda = df_processos.merge(
    features_eventos,
    on=PROCESS_ID_COL,
    how="left"
)

print("Processos:", df_processos.shape)
print("Features eventos:", features_eventos.shape)
print("ABT:", df_abt_eda.shape)

In [ ]:
assert len(df_abt_eda) == len(df_processos), "O merge alterou o número de linhas!"

df_abt_eda["flag_tem_eventos"] = df_abt_eda["qtd_eventos"].notna().astype(int)

event_count_cols = [
    c for c in df_abt_eda.columns
    if c.startswith("qtd_evento_cat_") or c.startswith("flag_teve_")
]

df_abt_eda[event_count_cols] = df_abt_eda[event_count_cols].fillna(0)

df_abt_eda.head()

In [ ]:
df_abt_eda.to_parquet(
    PROCESSED_DIR / "abt_eda_sem_target.parquet",
    index=False
)

## 30. Criar target inicial

In [ ]:
target_df = build_initial_target(df_abt_eda)

target_df.head()

In [ ]:
target_df[TARGET_COL].value_counts(dropna=False)

In [ ]:
df_abt_eda = df_abt_eda.merge(
    target_df,
    on=PROCESS_ID_COL,
    how="left"
)

df_abt_eda[TARGET_COL].value_counts(dropna=False, normalize=True)

In [ ]:
target_distribution = (
    df_abt_eda[TARGET_COL]
    .value_counts(dropna=False)
    .reset_index()
)

target_distribution.columns = [TARGET_COL, "qtd"]
target_distribution["perc"] = target_distribution["qtd"] / len(df_abt_eda)

target_distribution.to_csv(
    REPORTS_DIR / "target_distribution_inicial.csv",
    index=False
)

target_distribution

In [ ]:
df_abt_eda.to_parquet(
    PROCESSED_DIR / "abt_eda_inicial.parquet",
    index=False
)

## 31. Análise target × categorias

In [ ]:
df_model_eda = df_abt_eda[df_abt_eda[TARGET_COL].notna()].copy()
df_model_eda[TARGET_COL] = df_model_eda[TARGET_COL].astype(int)

print(df_model_eda.shape)
print("Taxa global de perda:", df_model_eda[TARGET_COL].mean())

In [ ]:
cols_to_analyze = [
    "assunto_normalizado_texto",
    "assunto_texto",
    "classe_texto",
    "uf",
    "cidade",
    "vara_texto",
    "orgao_julgador_texto",
    "fase_processual_texto",
    "ultima_categoria_evento",
]

for col in cols_to_analyze:
    if col in df_model_eda.columns:
        report = target_rate_by_category(
            df_model_eda,
            col,
            target_col=TARGET_COL,
            min_count=30
        )

        safe_col = col.replace("/", "_").replace(" ", "_")

        report.to_csv(
            REPORTS_DIR / f"target_rate_by_{safe_col}.csv",
            index=False
        )

        print("\n", "=" * 100)
        print(col)
        display(report.head(20))

## 32. Análise das flags de eventos contra target

In [ ]:
flag_analysis_df = analyze_event_flags_vs_target(df_model_eda)

flag_analysis_df

In [ ]:
flag_analysis_df.to_csv(
    REPORTS_DIR / "flag_event_analysis.csv",
    index=False
)

## 33. Validar saídas esperadas

In [ ]:
expected_outputs = [
    REPORTS_DIR / "overview_processos.csv",
    REPORTS_DIR / "null_report_processos.csv",
    REPORTS_DIR / "cardinality_report_processos.csv",
    REPORTS_DIR / "processos_profile_com_recomendacao.csv",
    REPORTS_DIR / "event_type_frequency_report.csv",
    REPORTS_DIR / "event_category_frequency_report.csv",
    REPORTS_DIR / "event_category_samples.csv",
    REPORTS_DIR / "target_distribution_inicial.csv",
    REPORTS_DIR / "flag_event_analysis.csv",
    PROCESSED_DIR / "eventos_tratados_eda.parquet",
    PROCESSED_DIR / "features_eventos_eda_inicial.parquet",
    PROCESSED_DIR / "abt_eda_inicial.parquet",
]

for path in expected_outputs:
    print(path, "OK" if path.exists() else "NÃO ENCONTRADO")

## 34. Ordem sugerida para leitura dos relatórios

Depois de rodar tudo, leia os relatórios nesta ordem:

1. `overview_processos.csv`
2. `auditoria_processos_eventos.csv`
3. `null_report_processos.csv`
4. `processos_profile_com_recomendacao.csv`
5. `event_type_frequency_report.csv`
6. `event_category_frequency_report.csv`
7. `event_category_samples.csv`
8. `target_distribution_inicial.csv`
9. `target_rate_by_assunto_normalizado_texto.csv`
10. `flag_event_analysis.csv`

## 35. Narrativa executiva da EDA

Conduzimos uma EDA forense para validar a qualidade e o potencial preditivo das bases de processos e eventos.

Primeiro verificamos o grão da base de processos, a integridade da chave e a relação entre processos e eventos. Em seguida, analisamos nulos, cardinalidade, valores dominantes, datas e colunas numéricas.

Depois classificamos as colunas do endpoint de processos em grupos de uso: candidatas a feature, possíveis targets, predições externas, campos deprecated, links, textos longos e colunas com risco de leakage.

Na base de eventos, normalizamos a coluna `tipo`, criamos macro categorias jurídicas e geramos uma primeira camada de features por processo, como quantidade de eventos, diversidade da trajetória, primeiro/último evento e flags de eventos relevantes.

Por fim, construímos uma ABT exploratória com uma linha por processo e um target inicial de perda do banco, permitindo avaliar quais assuntos, varas, fases e eventos têm maior associação com perda.